# Testbed results

Loads the newest run of each experiment from `results/` and summarizes it.

Start it from the repository root with:

```bash
uv run --group analysis jupyter lab analysis/results.ipynb
```

Then use **Run → Run All Cells**. Re-run after each experiment to refresh.

Each trial is one row in `trials.jsonl`. Columns are the configuration
(`nat_a`, `net_profile`, `pairing`, ...) plus whatever the trial measured.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

RESULTS = Path("../results")   # relative to this notebook (analysis/)

def latest_run(experiment: str) -> Path | None:
    """Newest results folder for an experiment, or None if it hasn't been run."""
    runs = sorted((RESULTS / experiment).glob("*/trials.jsonl"))
    return runs[-1].parent if runs else None

def load(experiment: str) -> pd.DataFrame | None:
    run = latest_run(experiment)
    if run is None:
        print(f"No results for '{experiment}' yet. Run: "
              f"uv run python -m p2prunner run experiments/{experiment}.toml")
        return None
    df = pd.read_json(run / "trials.jsonl", lines=True)
    print(f"{experiment}: {len(df)} trials from {run}")
    return df

# Chart style: thin marks, recessive grid, no chart junk.
SERIES = ["#2a78d6", "#eb6834"]          # categorical slots 1-2 (blue, orange)
INK, INK_2, GRID = "#0b0b0b", "#52514e", "#e4e3df"
plt.rcParams.update({
    "figure.dpi": 110, "axes.edgecolor": GRID, "axes.labelcolor": INK_2,
    "axes.titlecolor": INK, "axes.titleweight": "bold", "axes.titlesize": 11,
    "axes.grid": True, "axes.axisbelow": True, "grid.color": GRID, "grid.linewidth": 0.8,
    "axes.spines.top": False, "axes.spines.right": False,
    "xtick.color": INK_2, "ytick.color": INK_2, "legend.frameon": False,
})

## 1. NAT matrix: is every configuration what it claims to be?

From `experiments/nat_matrix.toml`. Every row should say **ok**: the measured
RFC 5780 mapping and filtering behaviour equals what the configuration should
produce. Codes follow RFC 4787: **EI** endpoint-independent,
**AD** address-dependent, **APD** address-and-port-dependent.

In [ ]:
nat = load("nat_matrix")
if nat is not None:
    cols = ["nat_a", "nat_b", "upstream_b"]
    table = (nat.groupby(cols)
                .agg(trials=("ok", "size"), passed=("ok", "sum"),
                     side_a=("mapping_a", lambda s: f"{s.iloc[0]}/{nat.loc[s.index[0], 'filtering_a']}"),
                     side_b=("mapping_b", lambda s: f"{s.iloc[0]}/{nat.loc[s.index[0], 'filtering_b']}"))
                .reset_index())
    table["result"] = (table.passed == table.trials).map({True: "ok", False: "MISMATCH"})
    print(f"{(table.result == 'ok').sum()} of {len(table)} configurations verified "
          "(side columns: mapping/filtering of the first trial)")
    display(table)

Any trial that did **not** match is listed below with what was measured, and
the verdict lines from the RFC 5780 tool's own output (saved under `raw/`).

In [ ]:
if nat is not None:
    import json
    bad = nat[~nat.ok.astype(bool)]
    if bad.empty:
        print("Every trial matched its expected behaviour.")
    else:
        run = latest_run("nat_matrix")
        cols = ["config_id", "trial"] + [f"{p}{k}_{s}" for s in "ab"
                for p in ("", "expected_") for k in ("mapping", "filtering")]
        display(bad[[c for c in cols if c in bad.columns]])
        for _, row in bad.iterrows():
            raw_file = run / "raw" / row.config_id / f"{row.trial:05d}.json"
            raw = json.loads(raw_file.read_text()) if raw_file.exists() else {}
            print(f"\n{row.config_label}  (trial {row.trial})")
            for side in "ab":
                lines = [l for l in raw.get(f"natdiscovery_{side}", "").splitlines()
                         if l.startswith("NAT with") or "timeout" in l.lower()]
                print(f"  side {side.upper()}: " + ("; ".join(lines) or "(no verdict in output)"))

## 2. Path baseline: what does each setup cost before any of our code?

From `experiments/path_baseline.toml`. Each peer pings the STUN server through
its NAT(s) and the latency shaping. Each home router delays only its own
outgoing traffic, so the **expected** RTT to a WAN host is half the profile's
peer-to-peer RTT (dashed line). Each dot is one trial's mean RTT.

In [ ]:
PROFILE_ORDER = ["none", "local", "coast", "transatlantic"]
EXPECTED_MS = {"none": 0, "local": 5, "coast": 35, "transatlantic": 75}

path = load("path_baseline")
if path is not None:
    # One row per (trial, side) instead of a_/b_ columns side by side.
    long = pd.concat([
        path.assign(side=s.upper(),
                    rtt=path[f"{s}_rtt_ms_mean"], p95=path[f"{s}_rtt_ms_p95"],
                    jitter=path[f"{s}_rtt_ms_stdev"], loss=path[f"{s}_loss"])
        for s in "ab"
    ])[["pairing", "net_profile", "side", "rtt", "p95", "jitter", "loss"]]

    summary = (long.groupby(["net_profile", "pairing", "side"])
                   .agg(trials=("rtt", "size"), rtt_median=("rtt", "median"),
                        rtt_p95=("p95", "median"), jitter=("jitter", "median"),
                        loss=("loss", "mean"))
                   .reset_index())
    summary["expected"] = summary.net_profile.map(EXPECTED_MS)
    summary["net_profile"] = pd.Categorical(summary.net_profile, PROFILE_ORDER, ordered=True)
    display(summary.sort_values(["net_profile", "pairing", "side"]).round(2))

In [ ]:
if path is not None:
    profiles = [p for p in PROFILE_ORDER if p in set(long.net_profile)]
    pairings = sorted(long.pairing.unique(), key=lambda p: (p != "realistic", p))
    fig, axes = plt.subplots(1, 2, figsize=(10, 3.8), sharey=True)
    for ax, side in zip(axes, ["A", "B"]):
        for i, pairing in enumerate(pairings):
            for x, profile in enumerate(profiles):
                vals = long[(long.side == side) & (long.pairing == pairing)
                            & (long.net_profile == profile)].rtt
                # spread the pairings side by side within each profile
                offset = (i - (len(pairings) - 1) / 2) * 0.22
                ax.scatter([x + offset] * len(vals), vals, s=22, color=SERIES[i % 2],
                           alpha=0.75, linewidths=0, label=pairing if x == 0 else None)
        for x, profile in enumerate(profiles):
            ax.hlines(EXPECTED_MS[profile], x - 0.4, x + 0.4, colors=INK_2,
                      linestyles="dashed", linewidth=1)
        ax.set_xticks(range(len(profiles)), profiles)
        ax.set_title(f"Side {side}")
        ax.grid(axis="x", visible=False)   # categories need no vertical gridlines
        ax.set_xlim(-0.6, len(profiles) - 0.4)
    axes[0].set_ylabel("Mean RTT to WAN host (ms)")
    axes[0].legend(title=None, loc="upper left")
    fig.suptitle("Access-path RTT by distance profile (dashed = expected)", x=0.01,
                 ha="left", fontsize=12, fontweight="bold", color=INK)
    fig.tight_layout()
    plt.show()

## 3. Punch matrix: which NAT pairings can open a direct path?

From `experiments/punch_matrix.toml` (the raw UDP hole-punching baseline).
A trial **succeeds** when both peers punched (`a_punched` and `b_punched`);
`ok` only means the tool ran. Each cell is the success rate over all trials.

In [ ]:
punch = load("punch_matrix")
if punch is not None:
    punch["success"] = punch.a_punched.fillna(False).astype(bool) & punch.b_punched.fillna(False).astype(bool)
    order = ["full_cone", "restricted", "port_restricted", "symmetric"]
    for upstream, part in punch.groupby("upstream_b"):
        rate = (part.pivot_table(index="nat_a", columns="nat_b", values="success", aggfunc="mean")
                    .reindex(index=order, columns=order))
        print(f"\nSuccess rate, side B upstream = {upstream} (rows: side A NAT, columns: side B NAT)")
        display(rate.style.format("{:.0%}", na_rep="-"))

    ok = punch[punch.success]
    detail = (ok.groupby(["nat_a", "nat_b", "upstream_b"])
                .agg(trials=("success", "size"),
                     punch_ms=("a_punch_ms", "median"),
                     rtt_ms=("a_rtt_ms_p50", "median"),
                     switched_a=("a_peer_addr_changed", "mean"),
                     switched_b=("b_peer_addr_changed", "mean"))
                .round(2))
    print("Successful pairings: median punch time, median peer-to-peer RTT, and how often "
          "each side switched to the peer's real (peer-reflexive) address")
    display(detail)

## 4. Punch path: what the open direct path delivers

From `experiments/punch_path.toml`. Peer-to-peer RTT crosses both home routers,
so it should match the **full** profile RTT (about 10 / 70 / 150 ms).
Throughput is offered above the 50 Mbit/s cap, so `rx_ip_mbit` should sit just
under the cap (netem also counts the 14-byte Ethernet header: about 49.4).

In [ ]:
pp = load("punch_path")
if pp is not None:
    FULL_RTT_MS = {"none": 0, "local": 10, "coast": 70, "transatlantic": 150}
    done = pp[pp.a_punched.fillna(False).astype(bool) & pp.b_punched.fillna(False).astype(bool)]
    summary = (done.groupby(["net_profile", "pairing"])
                   .agg(trials=("trial", "size"),
                        rtt_ms=("a_rtt_ms_p50", "median"),
                        rtt_p95=("a_rtt_ms_p95", "median"),
                        ping_loss=("a_ping_loss", "mean"),
                        a_to_b_ip_mbit=("b_rx_ip_mbit", "median"),
                        a_to_b_loss=("b_rx_loss", "mean"),
                        b_to_a_ip_mbit=("a_rx_ip_mbit", "median"),
                        b_to_a_loss=("a_rx_loss", "mean"))
                   .reset_index())
    summary["expected_rtt"] = summary.net_profile.map(FULL_RTT_MS)
    summary["net_profile"] = pd.Categorical(summary.net_profile, PROFILE_ORDER, ordered=True)
    print(f"{len(done)} of {len(pp)} trials punched")
    display(summary.sort_values(["net_profile", "pairing"]).round(3))